## Gold Layer: Wikipedia Topic Analytics

This notebook creates dashboard-ready Gold tables from the cleaned Silver pageview data.

The goal of the Gold layer is to transform article-level Wikipedia pageviews into topic-level analytics tables that can be used for reporting and dashboarding.

### Project Context

The project analyzes public attention patterns using Wikimedia pageview data.

Pipeline so far:

- Bronze: raw Wikimedia pageview data loaded from hourly files
- Silver: cleaned article-level pageviews
- Gold: topic-level aggregations for analytics and dashboards

### Gold Layer Goals

In this notebook, we will create topic-based analytics tables.

Planned outputs:

- topic pageviews by date and access method
- top articles by topic
- topic summary metrics
- dashboard-ready tables for Databricks SQL

### Topics

The first version will focus on selected topics such as:

- 2026 FIFA World Cup
- Artificial Intelligence
- Climate Change
- Space & Astronomy
- Bitcoin / Crypto
- Ukraine

These topics are selected because they are relevant, recognizable, and likely to show different attention patterns over time.

### Why This Step Matters

The Silver table is clean, but it is still article-level data.

The Gold layer makes the data easier to analyze by grouping related Wikipedia pages into meaningful topics.

This turns raw pageviews into a clearer analytical story:

Which topics attract attention?
Which articles dominate each topic?
Which topics show the strongest spikes?
How does attention differ between desktop and mobile users?

### Input Table

`wiki_silver.pageviews_clean`

### Output Tables

`wiki_gold.topic_pageviews`

`wiki_gold.top_pages_by_topic`

`wiki_gold.topic_summary`

In [0]:
# Use the project catalog

spark.sql("USE CATALOG workspace")

SILVER_TABLE = "wiki_silver.pageviews_clean"

print("Using catalog: workspace")
print("Silver table:", SILVER_TABLE)

Using catalog: workspace
Silver table: wiki_silver.pageviews_clean


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast

silver_df = spark.table(SILVER_TABLE)

print("Silver rows:", silver_df.count())

display(
    silver_df
    .orderBy(F.desc("views"))
    .limit(10)
)

Silver rows: 24489581


view_timestamp,view_date,source_date,source_hour,project,access_method,page_title,page_title_clean,views,source_url
2026-06-14T21:00:00.000Z,2026-06-14,2026-06-14,21,en.m,mobile,Zion_Suzuki,Zion Suzuki,292982,https://dumps.wikimedia.org/other/pageviews/2026/2026-06/pageviews-20260614-210000.gz
2026-06-14T20:00:00.000Z,2026-06-14,2026-06-14,20,en.m,mobile,Oliver_Tree,Oliver Tree,285495,https://dumps.wikimedia.org/other/pageviews/2026/2026-06/pageviews-20260614-200000.gz
2026-06-14T21:00:00.000Z,2026-06-14,2026-06-14,21,en.m,mobile,Oliver_Tree,Oliver Tree,249686,https://dumps.wikimedia.org/other/pageviews/2026/2026-06/pageviews-20260614-210000.gz
2026-06-14T22:00:00.000Z,2026-06-14,2026-06-14,22,en.m,mobile,Oliver_Tree,Oliver Tree,207934,https://dumps.wikimedia.org/other/pageviews/2026/2026-06/pageviews-20260614-220000.gz
2026-06-14T22:00:00.000Z,2026-06-14,2026-06-14,22,en.m,mobile,Zion_Suzuki,Zion Suzuki,163071,https://dumps.wikimedia.org/other/pageviews/2026/2026-06/pageviews-20260614-220000.gz
2026-06-14T23:00:00.000Z,2026-06-14,2026-06-14,23,en.m,mobile,Oliver_Tree,Oliver Tree,160781,https://dumps.wikimedia.org/other/pageviews/2026/2026-06/pageviews-20260614-230000.gz
2026-06-15T03:00:00.000Z,2026-06-15,2026-06-15,3,en.m,mobile,Yasin_Ayari,Yasin Ayari,134463,https://dumps.wikimedia.org/other/pageviews/2026/2026-06/pageviews-20260615-030000.gz
2026-06-15T00:00:00.000Z,2026-06-15,2026-06-15,0,en.m,mobile,Oliver_Tree,Oliver Tree,125887,https://dumps.wikimedia.org/other/pageviews/2026/2026-06/pageviews-20260615-000000.gz
2026-06-15T01:00:00.000Z,2026-06-15,2026-06-15,1,en.m,mobile,Oliver_Tree,Oliver Tree,107364,https://dumps.wikimedia.org/other/pageviews/2026/2026-06/pageviews-20260615-010000.gz
2026-06-15T02:00:00.000Z,2026-06-15,2026-06-15,2,en.m,mobile,Oliver_Tree,Oliver Tree,95590,https://dumps.wikimedia.org/other/pageviews/2026/2026-06/pageviews-20260615-020000.gz


In [0]:
# Define topic matching rules for the first Gold layer version

topic_rules = [
    (
        "Artificial Intelligence",
        r"\bartificial intelligence\b|"
        r"\bmachine learning\b|"
        r"\bdeep learning\b|"
        r"\bgenerative ai\b|"
        r"\blarge language models?\b|"
        r"\bneural networks?\b|"
        r"\bfoundation models?\b|"
        r"\bopenai\b|"
        r"\bchatgpt\b|"
        r"\bgpt-[2345]\b|"
        r"\banthropic\b|"
        r"\bclaude ai\b|"
        r"\bclaude chatbot\b|"
        r"\bmistral ai\b|"
        r"\bmixtral\b|"
        r"\bmistral 7b\b|"
        r"\bmistral large\b|"
        r"\bdeepseek\b|"
        r"\bgoogle gemini\b|"
        r"\bgemini ai\b|"
        r"\bgoogle deepmind\b|"
        r"\bmeta ai\b|"
        r"\bllama [234]\b|"
        r"\bllama language model\b|"
        r"\bxai\b|"
        r"\bgrok chatbot\b|"
        r"\bperplexity ai\b|"
        r"\bcohere\b|"
        r"\bhugging face\b"
    ),
    
    (
        "Climate Change",
        r"\bclimate change\b|"
        r"\bclimate crisis\b|"
        r"\bglobal warming\b|"
        r"\bgreenhouse effect\b|"
        r"\bgreenhouse gases?\b|"
        r"\bcarbon emissions?\b|"
        r"\bcarbon dioxide emissions?\b|"
        r"\bmethane emissions?\b|"
        r"\bcarbon footprint\b|"
        r"\bcarbon neutrality\b|"
        r"\bnet zero\b|"
        r"\bdecarboni[sz]ation\b|"
        r"\bclimate mitigation\b|"
        r"\bclimate adaptation\b|"
        r"\bclimate policy\b|"
        r"\bclimate action\b|"
        r"\bclimate justice\b|"
        r"\bclimate engineering\b|"
        r"\bsea level rise\b|"
        r"\bextreme weather\b|"
        r"\bparis agreement\b|"
        r"\bkyoto protocol\b|"
        r"\bintergovernmental panel on climate change\b|"
        r"\bipcc\b|"
        r"\bunfccc\b|"
        r"\bemissions trading\b|"
        r"\bcarbon pricing\b|"
        r"\bcarbon tax\b|"
        r"\bcarbon capture\b|"
        r"\bfossil fuels?\b|"
        r"\bgreta thunberg\b|"
        r"\bclimate movement\b|"
        r"\bclimate strike\b"
    ),


    (
        "Space & Astronomy",
        r"\bspace exploration\b|"
        r"\bnasa\b|"
        r"\bspacex\b|"
        r"\bartemis program\b|"
        r"\bjames webb space telescope\b|"
        r"\bexoplanets?\b|"
        r"\bblack holes?\b|"
        r"\bstarship\b|"
        r"\bmars rover\b|"
        r"\bmoon landing\b"
    ),
    (
        "Bitcoin / Crypto",
        r"\bbitcoin\b|"
        r"\bcryptocurrenc(?:y|ies)\b|"
        r"\bethereum\b|"
        r"\bblockchain\b|"
        r"\bdogecoin\b|"
        r"\bcrypto market\b|"
        r"\bcrypto exchange\b|"
        r"\bcrypto wallet\b|"
        r"\bcrypto asset\b|"
        r"\bcrypto trading\b"
    ),
    (
        "2026 FIFA World Cup",
        r"\b2026 fifa world cup\b|"
        r"\bfifa world cup 2026\b|"
        r"\b2026 world cup\b|"
        r"\b2026 fifa world cup qualification\b|"
        r"\b2026 fifa world cup squads?\b|"
        r"\b2026 fifa world cup venues?\b|"
        r"\b2026 fifa world cup group [a-l]\b|"
        r"\b2026 fifa world cup final\b|"
        r"\b2026 fifa world cup knockout stage\b|"
        r"\b2026 fifa world cup broadcasting rights\b|"
        r"\b2026 fifa world cup statistics\b"
    ),

        
    (
        "Ukraine",
        r"\bukraine\b|"
        r"\brusso-ukrainian war\b|"
        r"\brussian invasion of ukraine\b|"
        r"\bwar in ukraine\b|"
        r"\bannexation of crimea\b|"
        r"\bcrimean crisis\b|"
        r"\bwar in donbas\b|"
        r"\bdonbas war\b|"
        r"\bdonetsk\b|"
        r"\bluhansk\b|"
        r"\bcrimea\b|"
        r"\bkyiv\b|"
        r"\bkharkiv\b|"
        r"\bkherson\b|"
        r"\bmariupol\b|"
        r"\bbakhmut\b|"
        r"\bzaporizhzhia\b|"
        r"\bbucha massacre\b|"
        r"\bbattle of kyiv\b|"
        r"\bbattle of mariupol\b|"
        r"\bbattle of bakhmut\b|"
        r"\bvolodymyr zelensk(?:y|yy)\b|"
        r"\barmed forces of ukraine\b|"
        r"\bukrainian air force\b|"
        r"\bukrainian refugee crisis\b|"
        r"\bblack sea grain initiative\b|"
        r"\bzaporizhzhia nuclear power plant\b"
    
    )
]

topic_rules_df = spark.createDataFrame(
    topic_rules,
    ["topic", "topic_regex"]
)

display(topic_rules_df)

topic,topic_regex
Artificial Intelligence,\bartificial intelligence\b|\bmachine learning\b|\bdeep learning\b|\bgenerative ai\b|\blarge language models?\b|\bneural networks?\b|\bfoundation models?\b|\bopenai\b|\bchatgpt\b|\bgpt-[2345]\b|\banthropic\b|\bclaude ai\b|\bclaude chatbot\b|\bmistral ai\b|\bmixtral\b|\bmistral 7b\b|\bmistral large\b|\bdeepseek\b|\bgoogle gemini\b|\bgemini ai\b|\bgoogle deepmind\b|\bmeta ai\b|\bllama [234]\b|\bllama language model\b|\bxai\b|\bgrok chatbot\b|\bperplexity ai\b|\bcohere\b|\bhugging face\b
Climate Change,\bclimate change\b|\bclimate crisis\b|\bglobal warming\b|\bgreenhouse effect\b|\bgreenhouse gases?\b|\bcarbon emissions?\b|\bcarbon dioxide emissions?\b|\bmethane emissions?\b|\bcarbon footprint\b|\bcarbon neutrality\b|\bnet zero\b|\bdecarboni[sz]ation\b|\bclimate mitigation\b|\bclimate adaptation\b|\bclimate policy\b|\bclimate action\b|\bclimate justice\b|\bclimate engineering\b|\bsea level rise\b|\bextreme weather\b|\bparis agreement\b|\bkyoto protocol\b|\bintergovernmental panel on climate change\b|\bipcc\b|\bunfccc\b|\bemissions trading\b|\bcarbon pricing\b|\bcarbon tax\b|\bcarbon capture\b|\bfossil fuels?\b|\bgreta thunberg\b|\bclimate movement\b|\bclimate strike\b
Space & Astronomy,\bspace exploration\b|\bnasa\b|\bspacex\b|\bartemis program\b|\bjames webb space telescope\b|\bexoplanets?\b|\bblack holes?\b|\bstarship\b|\bmars rover\b|\bmoon landing\b
Bitcoin / Crypto,\bbitcoin\b|\bcryptocurrenc(?:y|ies)\b|\bethereum\b|\bblockchain\b|\bdogecoin\b|\bcrypto market\b|\bcrypto exchange\b|\bcrypto wallet\b|\bcrypto asset\b|\bcrypto trading\b
2026 FIFA World Cup,\b2026 fifa world cup\b|\bfifa world cup 2026\b|\b2026 world cup\b|\b2026 fifa world cup qualification\b|\b2026 fifa world cup squads?\b|\b2026 fifa world cup venues?\b|\b2026 fifa world cup group [a-l]\b|\b2026 fifa world cup final\b|\b2026 fifa world cup knockout stage\b|\b2026 fifa world cup broadcasting rights\b|\b2026 fifa world cup statistics\b
Ukraine,\bukraine\b|\brusso-ukrainian war\b|\brussian invasion of ukraine\b|\bwar in ukraine\b|\bannexation of crimea\b|\bcrimean crisis\b|\bwar in donbas\b|\bdonbas war\b|\bdonetsk\b|\bluhansk\b|\bcrimea\b|\bkyiv\b|\bkharkiv\b|\bkherson\b|\bmariupol\b|\bbakhmut\b|\bzaporizhzhia\b|\bbucha massacre\b|\bbattle of kyiv\b|\bbattle of mariupol\b|\bbattle of bakhmut\b|\bvolodymyr zelensk(?:y|yy)\b|\barmed forces of ukraine\b|\bukrainian air force\b|\bukrainian refugee crisis\b|\bblack sea grain initiative\b|\bzaporizhzhia nuclear power plant\b


In [0]:
# Match each unique Wikipedia page title against the topic rules only once.
# The resulting page-level matches are materialized as a Delta table because
# DataFrame caching is not supported on Databricks Serverless compute.

TOPIC_MATCHES_TABLE = "wiki_gold.topic_matches"

title_topic_map_df = (
    silver_df
    .select("page_title_clean")
    .distinct()
    .withColumn(
        "page_title_lower",
        F.lower(F.col("page_title_clean"))
    )
    .crossJoin(topic_rules_df)
    .filter(
        F.expr("page_title_lower RLIKE topic_regex")
    )
    .select(
        "page_title_clean",
        "topic"
    )
)

topic_matches_build_df = (
    silver_df
    .join(
        title_topic_map_df,
        on="page_title_clean",
        how="inner"
    )
    .select(
        "view_timestamp",
        "view_date",
        "source_hour",
        "access_method",
        "page_title",
        "page_title_clean",
        "views",
        "topic"
    )
)

(
    topic_matches_build_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TOPIC_MATCHES_TABLE)
)

print(f"Gold helper table saved: {TOPIC_MATCHES_TABLE}")

topic_matches_df = spark.table(TOPIC_MATCHES_TABLE)

display(
    topic_matches_df
    .orderBy(F.desc("views"))
    .limit(50)
)

Gold helper table saved: wiki_gold.topic_matches


view_timestamp,view_date,source_hour,access_method,page_title,page_title_clean,views,topic
2026-06-14T21:00:00.000Z,2026-06-14,21,mobile,2026_FIFA_World_Cup,2026 FIFA World Cup,45388,2026 FIFA World Cup
2026-06-14T22:00:00.000Z,2026-06-14,22,mobile,2026_FIFA_World_Cup,2026 FIFA World Cup,41006,2026 FIFA World Cup
2026-06-14T20:00:00.000Z,2026-06-14,20,mobile,2026_FIFA_World_Cup,2026 FIFA World Cup,36705,2026 FIFA World Cup
2026-06-14T23:00:00.000Z,2026-06-14,23,mobile,2026_FIFA_World_Cup,2026 FIFA World Cup,30264,2026 FIFA World Cup
2026-06-15T02:00:00.000Z,2026-06-15,2,mobile,2026_FIFA_World_Cup,2026 FIFA World Cup,19359,2026 FIFA World Cup
2026-06-15T00:00:00.000Z,2026-06-15,0,mobile,2026_FIFA_World_Cup,2026 FIFA World Cup,19023,2026 FIFA World Cup
2026-06-15T03:00:00.000Z,2026-06-15,3,mobile,2026_FIFA_World_Cup,2026 FIFA World Cup,18974,2026 FIFA World Cup
2026-06-15T04:00:00.000Z,2026-06-15,4,mobile,2026_FIFA_World_Cup,2026 FIFA World Cup,17952,2026 FIFA World Cup
2026-06-15T05:00:00.000Z,2026-06-15,5,mobile,2026_FIFA_World_Cup,2026 FIFA World Cup,17627,2026 FIFA World Cup
2026-06-15T01:00:00.000Z,2026-06-15,1,mobile,2026_FIFA_World_Cup,2026 FIFA World Cup,17576,2026 FIFA World Cup


In [0]:
# Check matched views by topic

display(
    topic_matches_df
    .groupBy("topic")
    .agg(
        F.count("*").alias("matched_rows"),
        F.sum("views").alias("total_views")
    )
    .orderBy(F.desc("total_views"))
)

topic,matched_rows,total_views
2026 FIFA World Cup,1878,709128
Artificial Intelligence,4168,98586
Ukraine,16195,73539
Space & Astronomy,5694,41560
Climate Change,2724,7705
Bitcoin / Crypto,1321,5977


In [0]:
# Create the first Gold table: topic-level pageviews by timestamp and access method

TOPIC_PAGEVIEWS_TABLE = "wiki_gold.topic_pageviews"

topic_pageviews_df = (
    topic_matches_df
    .groupBy(
        "view_timestamp",
        "view_date",
        "source_hour",
        "topic",
        "access_method"
    )
    .agg(
        F.sum("views").alias("total_views"),
        F.countDistinct("page_title_clean").alias("matched_articles")
    )
    .orderBy(
        "view_timestamp",
        "topic",
        "access_method"
    )
)

display(topic_pageviews_df)

view_timestamp,view_date,source_hour,topic,access_method,total_views,matched_articles
2026-06-14T20:00:00.000Z,2026-06-14,20,2026 FIFA World Cup,desktop,24959,79
2026-06-14T20:00:00.000Z,2026-06-14,20,2026 FIFA World Cup,mobile,60851,79
2026-06-14T20:00:00.000Z,2026-06-14,20,Artificial Intelligence,desktop,3331,220
2026-06-14T20:00:00.000Z,2026-06-14,20,Artificial Intelligence,mobile,5765,150
2026-06-14T20:00:00.000Z,2026-06-14,20,Bitcoin / Crypto,desktop,257,62
2026-06-14T20:00:00.000Z,2026-06-14,20,Bitcoin / Crypto,mobile,301,52
2026-06-14T20:00:00.000Z,2026-06-14,20,Climate Change,desktop,382,132
2026-06-14T20:00:00.000Z,2026-06-14,20,Climate Change,mobile,350,99
2026-06-14T20:00:00.000Z,2026-06-14,20,Space & Astronomy,desktop,1697,270
2026-06-14T20:00:00.000Z,2026-06-14,20,Space & Astronomy,mobile,2380,247


In [0]:
topic_pageviews_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TOPIC_PAGEVIEWS_TABLE)

print(f"Gold table saved: {TOPIC_PAGEVIEWS_TABLE}")

Gold table saved: wiki_gold.topic_pageviews


In [0]:
spark.sql("""
SELECT
  topic,
  access_method,
  SUM(total_views) AS total_views,
  SUM(matched_articles) AS article_hour_matches
FROM wiki_gold.topic_pageviews
GROUP BY topic, access_method
ORDER BY total_views DESC
""").show(truncate=False)

+-----------------------+-------------+-----------+--------------------+
|topic                  |access_method|total_views|article_hour_matches|
+-----------------------+-------------+-----------+--------------------+
|2026 FIFA World Cup    |mobile       |477109     |925                 |
|2026 FIFA World Cup    |desktop      |232019     |953                 |
|Artificial Intelligence|mobile       |60080      |1703                |
|Ukraine                |mobile       |42898      |7805                |
|Artificial Intelligence|desktop      |38506      |2465                |
|Ukraine                |desktop      |30641      |8390                |
|Space & Astronomy      |mobile       |23277      |2767                |
|Space & Astronomy      |desktop      |18283      |2927                |
|Climate Change         |desktop      |4428       |1671                |
|Climate Change         |mobile       |3277       |1053                |
|Bitcoin / Crypto       |desktop      |3034       |

In [0]:
from pyspark.sql.window import Window

TOP_PAGES_BY_TOPIC_TABLE = "wiki_gold.top_pages_by_topic"

top_pages_base_df = (
    topic_matches_df
    .groupBy(
        "topic",
        "page_title_clean"
    )
    .agg(
        F.sum("views").alias("total_views"),
        F.sum(F.when(F.col("access_method") == "mobile", F.col("views")).otherwise(0)).alias("mobile_views"),
        F.sum(F.when(F.col("access_method") == "desktop", F.col("views")).otherwise(0)).alias("desktop_views")
    )
)

topic_rank_window = Window.partitionBy("topic").orderBy(F.desc("total_views"))

top_pages_by_topic_df = (
    top_pages_base_df
    .withColumn("page_rank", F.row_number().over(topic_rank_window))
    .filter(F.col("page_rank") <= 20)
    .select(
        "topic",
        "page_rank",
        "page_title_clean",
        "total_views",
        "mobile_views",
        "desktop_views"
    )
    .orderBy("topic", "page_rank")
)

display(top_pages_by_topic_df)

topic,page_rank,page_title_clean,total_views,mobile_views,desktop_views
2026 FIFA World Cup,1,2026 FIFA World Cup,422826,290890,131936
2026 FIFA World Cup,2,2026 FIFA World Cup squads,31622,17383,14239
2026 FIFA World Cup,3,2026 FIFA World Cup Group F,25081,15357,9724
2026 FIFA World Cup,4,2026 FIFA World Cup qualification,23721,16132,7589
2026 FIFA World Cup,5,2026 FIFA World Cup knockout stage,20569,14070,6499
2026 FIFA World Cup,6,2026 FIFA World Cup Group E,16407,9230,7177
2026 FIFA World Cup,7,2026 FIFA World Cup qualification (AFC),13595,10142,3453
2026 FIFA World Cup,8,2026 FIFA World Cup final,13310,11262,2048
2026 FIFA World Cup,9,2026 FIFA World Cup qualification (CONCACAF),10439,8018,2421
2026 FIFA World Cup,10,2026 FIFA World Cup qualification (CONMEBOL),9134,6569,2565


In [0]:
top_pages_by_topic_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TOP_PAGES_BY_TOPIC_TABLE)

print(f"Gold table saved: {TOP_PAGES_BY_TOPIC_TABLE}")

Gold table saved: wiki_gold.top_pages_by_topic


In [0]:
display(
    spark.table("wiki_gold.top_pages_by_topic")
    .orderBy("topic", "page_rank")
)

topic,page_rank,page_title_clean,total_views,mobile_views,desktop_views
2026 FIFA World Cup,1,2026 FIFA World Cup,422826,290890,131936
2026 FIFA World Cup,2,2026 FIFA World Cup squads,31622,17383,14239
2026 FIFA World Cup,3,2026 FIFA World Cup Group F,25081,15357,9724
2026 FIFA World Cup,4,2026 FIFA World Cup qualification,23721,16132,7589
2026 FIFA World Cup,5,2026 FIFA World Cup knockout stage,20569,14070,6499
2026 FIFA World Cup,6,2026 FIFA World Cup Group E,16407,9230,7177
2026 FIFA World Cup,7,2026 FIFA World Cup qualification (AFC),13595,10142,3453
2026 FIFA World Cup,8,2026 FIFA World Cup final,13310,11262,2048
2026 FIFA World Cup,9,2026 FIFA World Cup qualification (CONCACAF),10439,8018,2421
2026 FIFA World Cup,10,2026 FIFA World Cup qualification (CONMEBOL),9134,6569,2565


In [0]:
# Create the third Gold table: topic summary metrics

TOPIC_SUMMARY_TABLE = "wiki_gold.topic_summary"

topic_summary_df = (
    topic_matches_df
    .groupBy("topic")
    .agg(
        F.sum("views").alias("total_views"),
        F.countDistinct("page_title_clean").alias("distinct_matched_articles"),
        F.sum(F.when(F.col("access_method") == "mobile", F.col("views")).otherwise(0)).alias("mobile_views"),
        F.sum(F.when(F.col("access_method") == "desktop", F.col("views")).otherwise(0)).alias("desktop_views")
    )
    .withColumn(
        "mobile_share",
        F.round(F.col("mobile_views") / F.col("total_views") * 100, 1)
    )
    .withColumn(
        "desktop_share",
        F.round(F.col("desktop_views") / F.col("total_views") * 100, 1)
    )
    .orderBy(F.desc("total_views"))
)

display(topic_summary_df)

topic,total_views,distinct_matched_articles,mobile_views,desktop_views,mobile_share,desktop_share
2026 FIFA World Cup,709128,102,477109,232019,67.3,32.7
Artificial Intelligence,98586,553,60080,38506,60.9,39.1
Ukraine,73539,3625,42898,30641,58.3,41.7
Space & Astronomy,41560,809,23277,18283,56.0,44.0
Climate Change,7705,671,3277,4428,42.5,57.5
Bitcoin / Crypto,5977,183,2943,3034,49.2,50.8


In [0]:
topic_summary_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TOPIC_SUMMARY_TABLE)

print(f"Gold table saved: {TOPIC_SUMMARY_TABLE}")

Gold table saved: wiki_gold.topic_summary


In [0]:
display(
    spark.table("wiki_gold.topic_summary")
    .orderBy(F.desc("total_views"))
)

topic,total_views,distinct_matched_articles,mobile_views,desktop_views,mobile_share,desktop_share
2026 FIFA World Cup,709128,102,477109,232019,67.3,32.7
Artificial Intelligence,98586,553,60080,38506,60.9,39.1
Ukraine,73539,3625,42898,30641,58.3,41.7
Space & Astronomy,41560,809,23277,18283,56.0,44.0
Climate Change,7705,671,3277,4428,42.5,57.5
Bitcoin / Crypto,5977,183,2943,3034,49.2,50.8


In [0]:
display(
    spark.table("wiki_gold.topic_pageviews")
    .orderBy("topic", "access_method")
)

view_timestamp,view_date,source_hour,topic,access_method,total_views,matched_articles
2026-06-14T22:00:00.000Z,2026-06-14,22,2026 FIFA World Cup,desktop,21172,79
2026-06-15T07:00:00.000Z,2026-06-15,7,2026 FIFA World Cup,desktop,17862,79
2026-06-15T04:00:00.000Z,2026-06-15,4,2026 FIFA World Cup,desktop,17817,77
2026-06-15T05:00:00.000Z,2026-06-15,5,2026 FIFA World Cup,desktop,20272,82
2026-06-15T01:00:00.000Z,2026-06-15,1,2026 FIFA World Cup,desktop,15534,78
2026-06-14T21:00:00.000Z,2026-06-14,21,2026 FIFA World Cup,desktop,21322,80
2026-06-14T23:00:00.000Z,2026-06-14,23,2026 FIFA World Cup,desktop,21592,81
2026-06-14T20:00:00.000Z,2026-06-14,20,2026 FIFA World Cup,desktop,24959,79
2026-06-15T06:00:00.000Z,2026-06-15,6,2026 FIFA World Cup,desktop,16635,82
2026-06-15T00:00:00.000Z,2026-06-15,0,2026 FIFA World Cup,desktop,16753,80
